# Semiconductor Quality Control - Logistic Regression Analysis
---
**Objective:** Sequential analysis of semiconductor defects using Logistic Regression, including regularization, polynomial features, and threshold tuning.


# 1. Imports and Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import make_scorer, fbeta_score, recall_score, precision_score, ConfusionMatrixDisplay, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Configuration
RANDOM_STATE = 1
FOLDS = 4
FILE_PATH = 'semiconductor_quality_control.csv'


# 2. Preliminaries
## 2.1 Load Data and Cleaning
Loading the dataset, dropping identifiers, and handling categorical variables.


In [ ]:
print(f">>> Loading Data from {FILE_PATH}...")
df = pd.read_csv(FILE_PATH)

# Drop identifiers and leakage columns (from log_reg_vanilla.py)
sybau = ['Process_ID', 'Timestamp', 'Wafer_ID', 'Defect', 'Join_Status']
x = df.drop(columns=sybau)
y = df['Defect']

# One-Hot Encoding for Categorical 'Tool_Type'
x = pd.get_dummies(x, columns=['Tool_Type'], drop_first=True)

print("Data Loaded. Shape:", x.shape)


## 2.2 Train/Test Split and Normalization
Standardizing numerical features.


In [ ]:
# Identify numerical columns for Scaling
numerical_cols = ['Chamber_Temperature', 'Gas_Flow_Rate', 'RF_Power', 'Etch_Depth',
                  'Rotation_Speed', 'Vacuum_Pressure', 'Stage_Alignment_Error',
                  'Vibration_Level', 'UV_Exposure_Intensity', 'Particle_Count']

# Split Data
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

# Scaling
scaler = StandardScaler()
scaler.fit(x_train[numerical_cols])
x_train.loc[:,numerical_cols] = scaler.transform(x_train[numerical_cols])
x_test.loc[:,numerical_cols] = scaler.transform(x_test[numerical_cols])

print('Pre-processing done!')


# 3. Exploratory Analysis
## 3.1 Feature Correlations
Analyzing which features have the strongest relationship with the Defect target.


In [ ]:
# Prepare data for correlation (Drop non-predictive IDs)
corr_drop_cols = ['Process_ID', 'Timestamp', 'Wafer_ID', 'Join_Status']
df_corr = df.drop(columns=corr_drop_cols)

# One-Hot Encode 'Tool_Type' for correlation check
df_corr = pd.get_dummies(df_corr, columns=['Tool_Type'], drop_first=True)

# Calculate correlations with Target
correlations = df_corr.corr()['Defect'].drop('Defect')

# Sort by magnitude (absolute value) and take top 15
top_features = correlations.abs().sort_values(ascending=False).head(15)
top_features_signed = correlations[top_features.index]

print("Top 15 Features by Correlation Magnitude:")
print(top_features_signed)


# 4. Baseline Modeling
## 4.1 Vanilla Logistic Regression
Training a standard logistic regression model with class balancing.


In [ ]:
# From log_reg_vanilla.py
model = LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE)
model.fit(x_train, y_train)
print('Vanilla Model Trained!')


## 4.2 Cross Validation Evaluation
Evaluating the baseline performance using Stratified K-Fold.


In [ ]:
# From log_reg_cross_val.py
pipeline = Pipeline([('scaler', StandardScaler()),('classifier', LogisticRegression(class_weight='balanced', C=1.0, max_iter=2000, random_state=1))])

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)
scoring = ['recall', 'f1', 'accuracy', 'precision']
scores = cross_validate(pipeline, x, y, cv=cv, scoring=scoring, return_train_score=False)

print(f"{'Fold':<6} {'Recall':<10} {'Precision':<10} {'F1':<10} {'Accuracy':<10}")
print("-" * 50)

for i in range(4):
    print(f"{i+1:<6} {scores['test_recall'][i]:<10.4f} {scores['test_precision'][i]:<10.4f} {scores['test_f1'][i]:<10.4f} {scores['test_accuracy'][i]:<10.4f}")

print("-" * 50)
print(f"Mean Recall: {np.mean(scores['test_recall']):.4f}")


# 5. Regularization
## 5.1 L1 (Lasso) vs L2 (Ridge) Grid Search
Iterating through penalty types and regularization strengths.


In [ ]:
# From log_reg_compiled.py
penalties = ['l1', 'l2']
C_values = np.logspace(-4, 4, 10)  # 10 values from 0.0001 to 10000
results = []

# Custom scorer for F2 (Beta=2 emphasizes Recall)
f2_scorer = make_scorer(fbeta_score, beta=2)

print(f"{'Penalty':<10} {'C':<12} {'F2':<10} {'Recall':<12}")
print("-" * 50)

for penalty in penalties:
    for C in C_values:
        # Solver selection based on penalty
        solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
        if penalty == 'l1': solver = 'liblinear' # force liblinear for l1 compatibility
        
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                penalty=penalty, 
                C=C, 
                solver=solver, 
                class_weight='balanced', 
                max_iter=5000, 
                random_state=RANDOM_STATE
            ))
        ])
        
        cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=RANDOM_STATE)
        
        f2_scores = cross_val_score(pipeline, x, y, cv=cv, scoring=f2_scorer)
        rec_scores = cross_val_score(pipeline, x, y, cv=cv, scoring='recall')
        
        mean_f2 = np.mean(f2_scores)
        mean_rec = np.mean(rec_scores)
        
        results.append({
            'Penalty': penalty,
            'C': C,
            'F2': mean_f2,
            'Recall': mean_rec
        })
        
        print(f"{penalty:<10} {C:<12.5f} {mean_f2:<10.4f} {mean_rec:<12.4f}")

results_df = pd.DataFrame(results)


## 5.2 Regularization Visualization
Plotting the impact of C values on Recall and F2 score.


In [ ]:
# From log_reg_regularization_var.py
plt.figure(figsize=(12, 7))

df_l1 = results_df[results_df['Penalty'] == 'l1']
df_l2 = results_df[results_df['Penalty'] == 'l2']

# --- Plot Recall ---
plt.plot(df_l1['C'], df_l1['Recall'], label='Recall (L1/Lasso)', 
         color='blue', marker='o', linestyle='-')
plt.plot(df_l2['C'], df_l2['Recall'], label='Recall (L2/Ridge)', 
         color='blue', marker='o', linestyle='--', alpha=0.6)

# --- Plot F2 Score ---
plt.plot(df_l1['C'], df_l1['F2'], label='F2 Score (L1/Lasso)', 
         color='red', marker='s', linestyle='-')
plt.plot(df_l2['C'], df_l2['F2'], label='F2 Score (L2/Ridge)', 
         color='red', marker='s', linestyle='--', alpha=0.6)

plt.xscale('log')
plt.xlabel('Inverse Regularization Strength (C)', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Impact of Regularization on Semiconductor Defect Detection', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, which="both", ls="-", alpha=0.4)
plt.show()


# 6. Advanced Feature Engineering
## 6.1 Polynomial Features
Expanding the feature space to capture interactions and non-linearities.


In [ ]:
# From log_reg_poly_features.py

categorical_cols = [c for c in x.columns if c not in numerical_cols]

# Create poly features on numerical columns only (using original train/test split data)
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
x_train_poly = poly.fit_transform(x_train[numerical_cols])
x_test_poly = poly.fit_transform(x_test[numerical_cols])

# Stack with categorical columns
x_train_final = np.hstack([x_train_poly, x_train[categorical_cols].values])
x_test_final = np.hstack([x_test_poly, x_test[categorical_cols].values])

print('Poly Features Created. New Shape:', x_train_final.shape)

# Train model on Poly Features
model_poly = LogisticRegression(class_weight='balanced', C=0.1, random_state=35)
model_poly.fit(x_train_final, y_train)
print('Poly Model Trained!')

y_pred_poly = model_poly.predict_proba(x_test_final)[:,1]


# 7. Optimization
## 7.1 Threshold Tuning ("Caramel")
Optimizing the decision threshold to maximize F2/Recall.


In [ ]:
# From log_reg_with_caramel.py

# Using probabilities from the Poly model (or we could use the Vanilla model)
y_pred = y_pred_poly # Using Poly predictions

best_score = 0
best_thresh = 0.5

print("Tuning Threshold...")
for t in np.arange(0.1,1.0,0.05):
    temp_pred = (y_pred >= t).astype(int)
    # Note: User code switched between f2 and recall. Using Recall as per last user script block.
    recall = recall_score(y_test, temp_pred)
    # print('Threshold:',t, 'Recall: ', recall)
    if recall > best_score: 
        best_score = recall
        best_thresh = t

print(f"Best Threshold: {best_thresh}")
print(f"Best Recall: {best_score}")


## 7.2 Winning Configuration
Final Summary of the best parameters found.


In [ ]:
# From log_reg_best.py
best_f2_idx = results_df['F2'].idxmax()
best_rec_idx = results_df['Recall'].idxmax()

print("\n" + "="*50)
print(" WINNING CONFIGURATIONS ")
print("="*50)
print(f"Best F2 Score:   {results_df.loc[best_f2_idx, 'F2']:.4f}")
print(f"  -> Params:     Penalty={results_df.loc[best_f2_idx, 'Penalty']}, C={results_df.loc[best_f2_idx, 'C']:.5f}")
print(f"Best Recall:     {results_df.loc[best_rec_idx, 'Recall']:.4f}")
print(f"  -> Params:     Penalty={results_df.loc[best_rec_idx, 'Penalty']}, C={results_df.loc[best_rec_idx, 'C']:.5f}")
